In [ ]:
import model
import epidemic_simulation
import survey_design
import random
import numpy as np
import scipy.stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.cm as cm
import arviz
import cmdstanpy
from cmdstanpy import CmdStanModel

In [ ]:
f_infectious, f_recovered, delay_inf, delay_recov = epidemic_simulation.delays()
transmission_rate = epidemic_simulation.transmission_rate

In [ ]:
inference_model = CmdStanModel(stan_file='model_rw2.stan', cpp_options={'STAN_THREADS':'true'})

sample_size = 8000
spacings = [21, 42, 84, 168]

full_sample_sizes = np.repeat(sample_size, len(spacings)**2)
prev_spacings = np.repeat(spacings, len(spacings))
sero_spacings = np.tile(spacings, len(spacings))

num_repeats = 10

dfs = []
all_scores = []
all_fits = []
all_expanded_scores = []
all_scores_energy = []
all_coverages = []
all_widths = []

In [ ]:
for _ in range(num_repeats):

    m = model.AgentModel(1000000, 50)

    surveys = survey_design.make_surveys(m, prev_spacings, full_sample_sizes, sero_spacings, full_sample_sizes, duration=7)
    df = survey_design.run_simulations(m, surveys, transmission_rate, f_infectious, f_recovered)

    fits = survey_design.run_inference(df, surveys, inference_model, delay_inf, delay_recov, 'overlapping', init_mcmc=800)
    scores, expanded_scores, scores_energy, coverage, width = survey_design.evaluate_fits(df, fits)

    dfs.append(df)
    all_scores.append(scores)
    all_fits.append(fits)
    all_expanded_scores.append(expanded_scores)

    all_scores_energy.append(scores_energy)

    all_coverages.append(coverage)
    all_widths.append(width)


In [ ]:

all_scores = np.asarray(all_scores)
avg_scores = np.median(all_scores, axis=0)

all_scores_energy = np.asarray(all_scores_energy)
avg_scores_energy = np.median(all_scores_energy, axis=0)

In [ ]:
fig = plt.figure(figsize=(5, 3.5))
ax = fig.add_subplot(1, 1, 1)


dense_spacing_prev = np.arange(16, 220, 0.25)
dense_spacing_sero = np.arange(16, 220, 0.25)

X, Y = np.meshgrid(dense_spacing_prev, dense_spacing_sero)

effort = np.zeros(X.shape)
effort_x = []
for i, prev_spacing in enumerate(dense_spacing_prev):
    for j, sero_spacing in enumerate(dense_spacing_sero):
        surveys = survey_design.make_surveys(m, [prev_spacing], [sample_size], [sero_spacing], [sample_size], duration=7)
        effort[j, i] = surveys[0].sample_size * len(surveys[0].start_dates) + surveys[1].sample_size * len(surveys[1].start_dates)

        if j == len(dense_spacing_sero) - 1:
            effort_x.append(effort[j, i])

levels = [60000, 100000, 140000, 180000, 220000, 260000]

true_levels = []
for l in levels:
    nearest = min(effort.flatten(), key=lambda x:abs(x-l))
    true_levels.append(nearest)

cs = ax.contour(X, Y, effort, true_levels, colors='tab:red', zorder=-10,)
ax.clabel(cs, cs.levels, fontsize=10)

cmap = mpl.colormaps['Greys_r']

highest_crps = max(avg_scores)
lowest_crps = min(avg_scores)

for score, spacing_prev, spacing_sero in zip(avg_scores, prev_spacings, sero_spacings):
    color = cmap((score - lowest_crps) / (highest_crps - lowest_crps))
    ax.scatter([spacing_prev,], [spacing_sero,], color=color, zorder=-1, s=300)
    ax.scatter([spacing_prev,], [spacing_sero,], color='k', zorder=-2, alpha=1, s=400)
    # ax.text(spacing_prev, spacing_sero, '{:.2f}'.format(score), color='red', zorder=100, ha='left')

cbar = plt.colorbar(cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=lowest_crps, vmax=highest_crps), cmap=cmap), ax=ax, label='Score')
cbar.set_ticks([lowest_crps, highest_crps])
cbar.set_ticklabels(['Best', 'Worst'])

ax.set_xlabel('Spacing between infection survey rounds (days)')
ax.set_ylabel('Spacing between serosurvey rounds (days)')

ax.set_yscale('log')
ax.set_xscale('log')

fig.set_tight_layout(True)

ax.set_xticks(spacings)
ax.set_yticks(spacings)
ax.set_xticklabels(spacings)
ax.set_yticklabels(spacings)

ax.set_xticks([], minor=True, )
ax.set_yticks([], minor=True)

fig.set_tight_layout(True)

plt.savefig('Figure5a.pdf')

plt.show()


In [ ]:
fig = plt.figure(figsize=(4.5, 2.5))
ax = fig.add_subplot(1, 1, 1)

for spacing in spacings:
    spacing_scores = all_scores[:, np.where(sero_spacings == spacing)]
    mid = np.mean(spacing_scores, axis=0)[0]
    ax.plot(spacings, mid, '.-', label=spacing)
    # std = np.std(spacing_scores, axis=0)[0]
    # ax.fill_between(spacings, mid - std, mid + std, alpha=0.25)

yticks = [np.min(all_scores), np.max(all_scores)]
ax.set_yticks(yticks)
ax.set_yticklabels(['Best', 'Worst'])

ax.spines[['right', 'top']].set_visible(False)
ax.legend(title='Sero. spacing', loc='center left', bbox_to_anchor=(1, 0.5))

ax.set_xlabel('Infection spacing')
ax.set_ylabel('Score')

fig.set_tight_layout(True)

plt.show()

In [ ]:
all_widths = np.asarray(all_widths)

fig = plt.figure(figsize=(6, 3.25))

ax = fig.add_subplot(1, 1, 1)

colors = {21: 'tab:blue', 42: 'tab:orange', 84: 'tab:green', 168: 'tab:red'}

sample_sizes_to_consider = {21: (0, -1), 42: (1, None), 84: (2, None)}
scores_plotted = []

for j, prev_spacing in enumerate(spacings):
    spacing_scores = all_scores[:, np.where(prev_spacings == prev_spacing)]
    mid = np.median(spacing_scores, axis=0)[0]

    # spacing_scores = all_scores[1, np.where(full_spacings == spacing)]
    # mid = np.mean(spacing_scores, axis=0)

    # s0, s1 = sample_sizes_to_consider[spacing]
    # mid = mid[s0:s1]
    # for i, sample_size in enumerate(sample_sizes[s0:s1]):
    for i, sero_spacing in enumerate(spacings):
        surveys = survey_design.make_surveys(m, [prev_spacing], [sample_size], [sero_spacing], [sample_size], duration=7)
        effort = surveys[0].sample_size * len(surveys[0].start_dates) + surveys[1].sample_size * len(surveys[1].start_dates)

        # ax.scatter([effort,], [mid[i],], color=colors[spacing], s=50, label=spacing if i == 0 else None)
        for k in range(spacing_scores.shape[0]):
            ax.scatter([effort,], [spacing_scores[k, 0, i],], color=colors[prev_spacing], s=25, label=prev_spacing if k == 0 and i == 0 else None, alpha=0.5)
            scores_plotted.append(spacing_scores[k, 0, i])


ax.legend(title='Spacing between inf.\nprev. rounds (days)', loc='center left', bbox_to_anchor=(1, 0.5))

yticks = [np.min(scores_plotted), np.max(scores_plotted)]
ax.set_yticks(yticks)
ax.set_yticklabels(['Best', 'Worst'])

ax.spines[['right', 'top']].set_visible(False)
ax.set_xlabel('Total tests conducted')
ax.set_ylabel('Score')

# ax.set_xscale('log')
# ax.set_yscale('log')

# ax.ticklabel_format(style='plain', axis='x')
plt.xticks(rotation=90, ha='center')

fig.set_tight_layout(True)

plt.savefig('Figure5b.pdf')

# ax.set_ylim(500, 17000)

plt.show()

In [ ]:
surveys = survey_design.make_surveys(m, prev_spacings, full_sample_sizes, sero_spacings, full_sample_sizes, duration=7)

fig = plt.figure(figsize=(7, 2.75), dpi=512)

idx1 = 3
idx2 = 15
idx3 = 0
idx4 = 12

dfi = 0
df = dfs[dfi]


ax = fig.add_subplot(2, 2, 1)

fit = all_fits[dfi][idx1]

for prev_surv_start in surveys[2 * idx1].start_dates:
    if prev_surv_start not in surveys[2 * idx1 + 1].start_dates:
        l1 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='yellowgreen')
    else:
        l2 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='gray')
for sero_surv_start in surveys[2 * idx1 + 1].start_dates:
    if sero_surv_start not in surveys[2 * idx1].start_dates:
        l3 = ax.axvspan(sero_surv_start - 3.5, sero_surv_start + 3.5, alpha=0.25, color='tab:orange')
    else:
        pass


ax.plot(df['transmissions'], color='k', label='Incidence')

x = np.asarray(fit.incidence[:, :])
lower = np.percentile(x, 5, axis=0,)
upper = np.percentile(x, 95, axis=0,)
mid = np.median(x, axis=0)
tplot = np.arange(len(mid))

ax.plot(tplot, mid,)
ax.fill_between(tplot, lower, upper, alpha=0.35)
ax.spines[['right', 'top']].set_visible(False)

ax.set_ylabel('Incidence')

ax.tick_params(labelbottom=False, labelleft=True)

ax.set_title('21 Days infection\nsurvey spacing')


ax = fig.add_subplot(2, 2, 2, sharex=ax, sharey=ax)

fit = all_fits[dfi][idx2]


for prev_surv_start in surveys[2 * idx2].start_dates:
    if prev_surv_start not in surveys[2 * idx2 + 1].start_dates:
        l1 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='yellowgreen')
    else:
        l2 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='gray')
for sero_surv_start in surveys[2 * idx2 + 1].start_dates:
    if sero_surv_start not in surveys[2 * idx2].start_dates:
        l3 = ax.axvspan(sero_surv_start - 3.5, sero_surv_start + 3.5, alpha=0.25, color='tab:orange')
    else:
        pass

ax.plot(df['transmissions'], color='k', label='Incidence')

x = np.asarray(fit.incidence[:, :])
lower = np.percentile(x, 5, axis=0,)
upper = np.percentile(x, 95, axis=0,)
mid = np.median(x, axis=0)
tplot = np.arange(len(mid))

ax.plot(tplot, mid,)
ax.fill_between(tplot, lower, upper, alpha=0.35)
ax.spines[['right', 'top']].set_visible(False)

legend_ax = ax

ax.set_title('168 Days infection\nsurvey spacing')

ax.tick_params(labelbottom=False, labelleft=False)



ax = fig.add_subplot(2, 2, 3, sharex=ax, sharey=ax)

fit = all_fits[dfi][idx3]


for prev_surv_start in surveys[2 * idx3].start_dates:
    if prev_surv_start not in surveys[2 * idx3 + 1].start_dates:
        l1 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.2, color='yellowgreen')
    else:
        l2 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.2, color='gray')
for sero_surv_start in surveys[2 * idx3 + 1].start_dates:
    if sero_surv_start not in surveys[2 * idx3].start_dates:
        l3 = ax.axvspan(sero_surv_start - 3.5, sero_surv_start + 3.5, alpha=0.25, color='tab:orange')
    else:
        pass

ax.plot(df['transmissions'], color='k', label='Incidence')

x = np.asarray(fit.incidence[:, :])
lower = np.percentile(x, 5, axis=0,)
upper = np.percentile(x, 95, axis=0,)
mid = np.median(x, axis=0)
tplot = np.arange(len(mid))

ax.plot(tplot, mid,)
ax.fill_between(tplot, lower, upper, alpha=0.35)
ax.spines[['right', 'top']].set_visible(False)


ax.set_ylabel('Incidence')
ax.set_xlabel('Time (days)')

ax.set_title('21 Days sero-\nsurvey spacing', x=-0.55, y=0.4)


ax = fig.add_subplot(2, 2, 4, sharex=ax, sharey=ax)

fit = all_fits[dfi][idx4]


for prev_surv_start in surveys[2 * idx4].start_dates:
    if prev_surv_start not in surveys[2 * idx4 + 1].start_dates:
        l1 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='yellowgreen')
    else:
        l2 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='gray')
for sero_surv_start in surveys[2 * idx4 + 1].start_dates:
    if sero_surv_start not in surveys[2 * idx4].start_dates:
        l3 = ax.axvspan(sero_surv_start - 3.5, sero_surv_start + 3.5, alpha=0.25, color='tab:orange')
    else:
        pass

l6, = ax.plot(df['transmissions'], color='k', label='Incidence')

x = np.asarray(fit.incidence[:, :])
lower = np.percentile(x, 5, axis=0,)
upper = np.percentile(x, 95, axis=0,)
mid = np.median(x, axis=0)
tplot = np.arange(len(mid))

l4, = ax.plot(tplot, mid,)
l5 = ax.fill_between(tplot, lower, upper, alpha=0.35)
ax.spines[['right', 'top']].set_visible(False)
ax.set_xlabel('Time (days)')

legend_ax.legend((l1, l2, l3, (l4, l5,), l6,), ['Infection survey', 'Combined survey', 'Serosurvey', 'Posterior', 'Incidence'], loc='center left', bbox_to_anchor=(1, 0.5))

ax.set_title('168 Days sero-\nsurvey spacing', x=-1.75, y=1.6)

ax.tick_params(labelbottom=True, labelleft=False)

# fig.set_tight_layout(True)

plt.savefig('Figure5c.pdf')

plt.show()